# Logistics Delivery Performance & Cost Analysis

A reproducible exploration of delivery timeliness, shipping spend, shipping modes, and delivery partners.

In [1]:
from pathlib import Path
import sys
import pandas as pd
import plotly.express as px

PROJECT_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents) if (path / 'run_pipeline.py').exists()),
    None,
)
if PROJECT_ROOT is None:
    candidate = Path.cwd() / 'logistics_delivery_analytics'
    PROJECT_ROOT = candidate if (candidate / 'run_pipeline.py').exists() else None
if PROJECT_ROOT is None:
    raise FileNotFoundError('Could not locate logistics_delivery_analytics/run_pipeline.py')
sys.path.insert(0, str(PROJECT_ROOT))

from src.data_loader import load_data
from src.data_cleaner import clean_data
from src.business_analysis import add_business_metrics, summarize_business

raw = load_data()
orders = add_business_metrics(clean_data(raw))
print(f'Raw rows: {len(raw):,} | Clean orders: {orders.order_id.nunique():,}')
orders.head()

Raw rows: 5,000 | Clean orders: 4,975


,order_id,customer_id,order_date,warehouse,origin_city,destination_city,distance_km,product_category,quantity,weight_kg,...,delivery_status,customer_rating,damage_flag,return_flag,delay_days,on_time,delivery_days,tracked_total_cost,shipping_cost_per_km,order_month
0,ORD101253,CUST12780,2025-03-12,Bengaluru,Bengaluru,Chennai,451.7,Home & Kitchen,7,5.51,...,Delayed,3.0,No,No,5,False,10,2426.22,4.526367,2025-03
1,ORD102838,CUST11842,2025-12-02,Delhi,Delhi,Surat,592.8,Automotive,1,1.05,...,Delivered,4.0,No,No,0,True,7,2967.72,4.356781,2025-12
2,ORD103457,CUST10940,2025-04-21,Pune,Pune,Nagpur,191.9,Clothing,3,9.09,...,Delivered,5.0,No,No,0,True,5,1003.06,4.258624,2025-04
3,ORD100061,CUST10263,2025-02-21,Pune,Pune,Coimbatore,376.4,Clothing,7,1.34,...,Delayed,3.0,No,No,2,False,8,2099.57,4.551594,2025-02
4,ORD103947,CUST12467,2025-08-22,Mumbai,Mumbai,Delhi,895.3,Sports,5,3.95,...,Delivered,4.0,No,No,0,True,6,4970.48,4.445504,2025-08


## Portfolio KPIs
Timeliness is calculated from actual and expected delivery dates, rather than inferred from the source status label.

In [2]:
kpis = summarize_business(orders)
pd.Series(kpis, name='value').to_frame()

,value
orders,4975.000000
on_time_rate_pct,76.100503
average_shipping_cost,2284.312187
average_tracked_total_cost,2656.235750
average_customer_rating,4.124642
average_delay_days_late_only,2.946173
damaged_order_rate_pct,2.854271
returned_order_rate_pct,4.361809


## Monthly delivery performance
The month is assigned by order date; each point is the share delivered on or before the expected date.

In [3]:
monthly = orders.groupby('order_month', as_index=False).agg(
    orders=('order_id', 'nunique'), on_time_rate=('on_time', 'mean')
)
figure = px.line(monthly, x='order_month', y='on_time_rate', markers=True, hover_data=['orders'],
                  title='On-time delivery by order month', color_discrete_sequence=['#087E6E'])
figure.update_yaxes(tickformat='.0%', title='On-time rate')
figure.update_xaxes(title='Order month')
figure.show()

## Delivery partner comparison
These are unadjusted averages and may reflect differences in distance, route, mode, and order mix.

In [4]:
partner_summary = orders.groupby('delivery_partner', as_index=False).agg(
    orders=('order_id', 'nunique'),
    on_time_rate=('on_time', 'mean'),
    avg_shipping_cost=('shipping_cost', 'mean'),
    avg_customer_rating=('customer_rating', 'mean'),
).sort_values('on_time_rate', ascending=False)
partner_summary

,delivery_partner,orders,on_time_rate,avg_shipping_cost,avg_customer_rating
0,Blue Dart,799,0.813517,2276.156591,4.118622
2,Delhivery,829,0.786490,2152.848808,4.140049
4,Shadowfax,841,0.785969,2344.868375,4.173171
3,Ecom Express,853,0.737397,2388.763491,4.137232
5,XpressBees,844,0.734597,2188.203441,4.077201
1,DTDC,809,0.709518,2354.670662,4.101124


## Shipping mode and cost
Tracked total cost adds the fuel field to shipping cost; confirm whether fuel is already included before interpreting this as an accounting total.

In [5]:
mode_summary = orders.groupby('shipping_mode', as_index=False).agg(
    orders=('order_id', 'nunique'),
    on_time_rate=('on_time', 'mean'),
    avg_shipping_cost=('shipping_cost', 'mean'),
    avg_tracked_total_cost=('tracked_total_cost', 'mean'),
).sort_values('orders', ascending=False)
mode_summary

,shipping_mode,orders,on_time_rate,avg_shipping_cost,avg_tracked_total_cost
3,Road,2521,0.744149,1844.261848,2215.652828
2,Rail,931,0.718582,1465.424733,1839.795593
0,Air,911,0.807903,3791.725437,4157.803848
1,Express,612,0.825163,3089.063832,3468.155000


## Data quality
Missing ratings and warehouse processing times remain missing. Fuel cost is imputed by shipping-mode median, then overall median; inspect the original source when reporting these measures.

In [6]:
quality = pd.DataFrame({
    'raw_missing': raw.isna().sum(),
    'cleaned_missing': clean_data(raw).isna().sum(),
})
quality.loc[quality.any(axis=1)].sort_values('raw_missing', ascending=False)

,raw_missing,cleaned_missing
customer_rating,89,89
warehouse_processing_hours,59,59
fuel_cost,50,0
shipping_cost,50,50
destination_city,40,40
